# Building Shor's Algorithm from Scratch, Part 2: The Modulo Adder

In [Part 1](401_shor_scratch_adder.ipynb) we built a plain quantum adder. This tutorial builds **modular** addition, $a+b \bmod N$, on top of it — the operation actually needed inside Shor's algorithm, since every intermediate value produced while computing $x^j \bmod N$ has to be kept within $[0, N)$.

Reference: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995), Figure 3.

## The five-step idea

$(a+b) \bmod N$ is easy classically: add, then subtract $N$ if the result is $\ge N$. On a quantum computer we can't just "check and branch" the usual way (that would require measuring, collapsing any superposition we care about) — instead the check has to be done *with a qubit*, and the conditional subtraction has to be done *with a controlled circuit*.

1. **Add**: $b \leftarrow a + b$ (using Part 1's adder)
2. **Subtract $N$**: $b \leftarrow b - N$ (the adder run backwards)
3. **Detect the sign**: if step 2 made $b$ go negative, the adder's extra top qubit ends up as $1$ (it acts like a borrow/overflow flag) — copy that bit into a flag qubit $t$
4. **Conditionally add $N$ back**: if $t=1$ (i.e. $a+b < N$, so subtracting $N$ overshot), add $N$ back — but *only* on the branches of the superposition where $t=1$, using $t$ as an extra control qubit
5. **Clean up**: restore $t$ to $\ket 0$ and undo the sign-detection step, so the whole thing is a clean, reusable, entangling-free-outside-$b$ operation

Steps 1-2 use the adder/subtractor directly; step 4 needs a *controlled* version of the whole adder circuit, which is the interesting new piece here.

In [1]:
!pip install git+https://github.com/blueqat/blueqatSDK

/Users/yuichirominato/.pyenv/shims/pip: line 8: /opt/homebrew/opt/pyenv/bin/pyenv: No such file or directory


## Reusing Part 1's building blocks

Same `carry`/`carry_inv`/`sum_gate` as before, plus the full adder assembled into `add_circuit`, and its inverse `sub_circuit` obtained simply by running the same gates in reverse order (every gate here is `CX` or `CCX`, both self-inverse, so reversing the *order* is enough — no need to invert any individual gate).

In [2]:
from blueqat import Circuit
from blueqat.gate import CXGate, ToffoliGate

def carry(c, a, b, c_next):
    circ = Circuit()
    circ.ccx[a, b, c_next]
    circ.cx[a, b]
    circ.ccx[c, b, c_next]
    return circ

def carry_inv(c, a, b, c_next):
    circ = Circuit()
    circ.ccx[c, b, c_next]
    circ.cx[a, b]
    circ.ccx[a, b, c_next]
    return circ

def sum_gate(c, a, b):
    circ = Circuit()
    circ.cx[a, b]
    circ.cx[c, b]
    return circ

def add_circuit(c, a, b, n):
    """b := a + b, using carry register c. b must have n+1 qubits."""
    circ = Circuit()
    for i in range(n - 1):
        circ += carry(c[i], a[i], b[i], c[i + 1])
    circ += carry(c[n - 1], a[n - 1], b[n - 1], b[n])
    circ.cx[a[n - 1], b[n - 1]]
    circ += sum_gate(c[n - 1], a[n - 1], b[n - 1])
    for i in range(n - 2, -1, -1):
        circ += carry_inv(c[i], a[i], b[i], c[i + 1])
        circ += sum_gate(c[i], a[i], b[i])
    return circ

def sub_circuit(c, a, b, n):
    """b := b - a: the adder's gate list, reversed."""
    add_circ = add_circuit(c, a, b, n)
    reversed_circ = Circuit()
    reversed_circ.ops = list(reversed(add_circ.ops))
    return reversed_circ

## Registers

The full modulo adder needs $4n+3$ qubits:

- `c[0..n-1]`: carry register (reused for both add and subtract steps)
- `a[0..n-1]`: first addend
- `b[0..n]`: second addend / running result (same $n+1$-wide register as Part 1 — its top bit doubles as the overflow/borrow flag during step 3)
- `N[0..n-1]`: the modulus, loaded as a constant
- `t`: one flag qubit for "did we need to add $N$ back?"
- one ancilla qubit, needed only to help build 3-controlled gates below

## Step 4 in detail: a controlled adder from an uncontrolled one

We already have `add_circuit(c, N, b, n)` for "$b \leftarrow b + N$" — but step 4 needs to apply it *only when $t=1$*, i.e. a version of every gate in that circuit controlled by $t$. Controlling a `CX` by one more qubit gives a `CCX` — easy. Controlling a `CCX` by one more qubit gives a **3-controlled X**, which isn't a gate this SDK has directly. The standard trick: use one ancilla qubit to compute the AND of the *first two* controls into the ancilla with a `CCX`, apply an ordinary `CCX` using the ancilla as one of its two controls, then uncompute the ancilla back to $\ket 0$ with the same `CCX` again. Net effect: a `CCX` controlled by one extra qubit, using only 2-control gates and one temporary qubit.

In [3]:
def build_modulo_adder(input_a, input_b, input_N, n):
    """b := (a + b) mod N. Returns (circuit, b_register_qubit_indices)."""
    num_qubits = 4 * n + 3
    circ = Circuit(num_qubits)

    c = list(range(n))
    a = list(range(n, 2 * n))
    b = list(range(2 * n, 3 * n + 1))
    N = list(range(3 * n + 1, 4 * n + 1))
    t = 4 * n + 1
    ancilla = 4 * n + 2

    # Encode inputs
    for i in range(n):
        if (input_a >> i) & 1:
            circ.x[a[i]]
        if (input_b >> i) & 1:
            circ.x[b[i]]
        if (input_N >> i) & 1:
            circ.x[N[i]]

    # Steps 1-2: b <- a + b, then b <- b - N
    circ += add_circuit(c, a, b, n)
    circ += sub_circuit(c, N, b, n)

    # Step 3: copy the borrow/overflow bit into the flag t
    circ.cx[b[n], t]

    # Step 4: if t == 1, add N back (t-controlled add_circuit, via the ancilla trick above)
    controlled_add = add_circuit(c, N, b, n)
    for g in controlled_add.ops:
        if isinstance(g, CXGate):
            circ.ccx[t, g.targets[0], g.targets[1]]
        elif isinstance(g, ToffoliGate):
            circ.ccx[t, g.targets[0], ancilla]
            circ.ccx[ancilla, g.targets[1], g.targets[2]]
            circ.ccx[t, g.targets[0], ancilla]

    # Step 5: clean up -- undo the borrow/overflow detection and reset t to |0>
    circ += sub_circuit(c, a, b, n)
    circ.x[b[n]]
    circ.cx[b[n], t]
    circ.x[b[n]]
    circ += add_circuit(c, a, b, n)

    return circ, b

## Testing it

Same bit-order convention as Part 1: qubit `q` is `state[N - 1 - q]` in the result string.

In [4]:
def run_modulo_adder(val_a, val_b, val_N, n):
    circuit, b = build_modulo_adder(val_a, val_b, val_N, n)
    state = circuit.m[:].run(shots=1).most_common(1)[0][0]
    total_qubits = len(state)
    bits = [state[total_qubits - 1 - q] for q in b]  # bits[0] = LSB
    return int("".join(reversed(bits)), 2)

In [5]:
tests = [
    (7, 6, 11, 4),
    (14, 13, 15, 4),
    (3, 5, 9, 4),
    (0, 0, 7, 3),
    (5, 5, 6, 3),
]
for x, y, N, n in tests:
    result = run_modulo_adder(x, y, N, n)
    expected = (x + y) % N
    print(f"({x}+{y}) mod {N} = {result}  (expected {expected}, {'OK' if result == expected else 'MISMATCH'})")

(7+6) mod 11 = 2  (expected 2, OK)


(14+13) mod 15 = 12  (expected 12, OK)


(3+5) mod 9 = 8  (expected 8, OK)


(0+0) mod 7 = 0  (expected 0, OK)


(5+5) mod 6 = 4  (expected 4, OK)


All five cases match. Note that a couple of these are chosen to exercise both branches: `(7+6) mod 11` actually needs the subtract-and-add-back correction (`7+6=13 \ge 11`), while `(0+0) mod 7` never goes negative and step 4 should do nothing — both work correctly with the same circuit.

## What's next

With addition and modular addition in hand, the next step is **[controlled modular multiplication](403_shor_scratch_controlled_multiplier.ipynb)** — multiplying by a fixed constant modulo $N$, controlled on a qubit, by repeatedly modulo-adding shifted copies of the multiplicand (binary long multiplication, done reversibly). That's the piece that finally lets us build modular *exponentiation*, and from there, the full algorithm.